<a href="https://colab.research.google.com/github/MyaSvelte/IBM-Prompt-Engineering-Lab/blob/main/pe_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install git+https://github.com/ibm-granite-community/utils \
    "langchain_community<0.3.0" \
    replicate

  Cloning https://github.com/ibm-granite-community/utils to /tmp/pip-req-build-l1zyzhh6
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite-community/utils /tmp/pip-req-build-l1zyzhh6
  Resolved https://github.com/ibm-granite-community/utils to commit 083172b3a3f5400ad852d7309cce1c381229008b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import os
import replicate
from langchain_community.llms import Replicate
from ibm_granite_community.notebook_utils import get_env_var
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML
from string import Template

In [4]:
#Use the utility function to get from environment or prompt
replicate_api_token = get_env_var("REPLICATE_API_TOKEN")
os.environ["REPLICATE_API_TOKEN"] = replicate_api_token

REPLICATE_API_TOKEN loaded from Google Colab secret.


In [5]:
# Model configuration - matching reference implementation
MODEL_NAME = "ibm-granite/granite-3.3-8b-instruct"
MAX_TOKENS = 1024
TEMPERATURE = 0.2

# Initialize the model
llm = Replicate(
    model=MODEL_NAME,
    model_kwargs={
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE
    }
)

print(f"✅ Model initialized: {MODEL_NAME}")
print(f"   Max Tokens: {MAX_TOKENS}")
print(f"   Temperature: {TEMPERATURE}")

✅ Model initialized: ibm-granite/granite-3.3-8b-instruct
   Max Tokens: 1024
   Temperature: 0.2


In [6]:
# Task 3: Draft an initial prompt for a specific project
# This is NOT a template yet - it's hard-coded for one specific use case

baseline_prompt = """Write a concise weekly status summary for the DeltaFin client project.
Highlight progress made this week, note any blockers, and outline next steps in a professional
tone suitable for a client email."""

print("🔵 TASK 3: Baseline Prompt (Single-Use)")
print("="*70)
print("\nPrompt:")
print(baseline_prompt)
print("\n" + "-"*70)
print("Generating output...\n")

# Generate output using the baseline prompt
baseline_output = llm.invoke(baseline_prompt)

print("Generated Output:")
print("-"*70)
print(baseline_output)
print("-"*70)

🔵 TASK 3: Baseline Prompt (Single-Use)

Prompt:
Write a concise weekly status summary for the DeltaFin client project.
Highlight progress made this week, note any blockers, and outline next steps in a professional
tone suitable for a client email.

----------------------------------------------------------------------
Generating output...

Generated Output:
----------------------------------------------------------------------
Subject: Weekly Status Update - DeltaFin Project

Dear [Client's Name],

I hope this message finds you well. I am writing to provide you with a concise update on the DeltaFin project's progress for this week.

**Progress Made:**

1. **Data Migration:** We successfully migrated 80% of the client data to the new system. The remaining 20% is currently undergoing validation to ensure accuracy before the final transfer.

2. **System Integration:** The integration of the DeltaFin platform with the existing infrastructure is progressing smoothly. We have completed the i

In [7]:
# Task 4: Convert the baseline prompt into a reusable template with variables

# Using Python's string formatting with named placeholders
prompt_template = """Write a concise weekly status summary for the {client_name} project.

Summarize progress made during {time_period}, note {blockers}, and outline {next_steps}
in a {tone} tone suitable for {audience}.

Additional context:
- Progress: {progress}
- Format: {output_format}
"""

print("🟢 TASK 4: Reusable Prompt Template")
print("="*70)
print("\nTemplate with Variables:")
print("-"*70)
print(prompt_template)
print("-"*70)

print("\n✅ Template created successfully!")
print("\nVariables identified:")
variables = ["{client_name}", "{time_period}", "{progress}", "{blockers}",
             "{next_steps}", "{tone}", "{audience}", "{output_format}"]
for var in variables:
    print(f"  • {var}")

🟢 TASK 4: Reusable Prompt Template

Template with Variables:
----------------------------------------------------------------------
Write a concise weekly status summary for the {client_name} project.

Summarize progress made during {time_period}, note {blockers}, and outline {next_steps}
in a {tone} tone suitable for {audience}.

Additional context:
- Progress: {progress}
- Format: {output_format}

----------------------------------------------------------------------

✅ Template created successfully!

Variables identified:
  • {client_name}
  • {time_period}
  • {progress}
  • {blockers}
  • {next_steps}
  • {tone}
  • {audience}
  • {output_format}


##Test Case 1: DeltaFin Project (Financial Services)

In [8]:
# Define variables for Project 1: DeltaFin
project1_vars = {
    "client_name": "DeltaFin",
    "time_period": "Week 4",
    "progress": "Completed API integration testing and resolved 15 critical bugs",
    "blockers": "testing delays caused by the external vendor's infrastructure issues",
    "next_steps": "finalize the integration plan and begin UAT preparation",
    "tone": "formal",
    "audience": "senior client stakeholders",
    "output_format": "structured paragraph with clear sections"
}

# Substitute variables into the template
project1_prompt = prompt_template.format(**project1_vars)

print("🟣 TASK 5: Testing Template - Project 1 (DeltaFin)")
print("="*70)
print("\nVariable Values:")
for key, value in project1_vars.items():
    print(f"  {key}: {value}")

print("\n" + "-"*70)
print("Generated Prompt:")
print("-"*70)
print(project1_prompt)

print("\n" + "-"*70)
print("Generating output...\n")

# Generate output
project1_output = llm.invoke(project1_prompt)

print("Generated Status Summary:")
print("-"*70)
print(project1_output)
print("-"*70)

🟣 TASK 5: Testing Template - Project 1 (DeltaFin)

Variable Values:
  client_name: DeltaFin
  time_period: Week 4
  progress: Completed API integration testing and resolved 15 critical bugs
  blockers: testing delays caused by the external vendor's infrastructure issues
  next_steps: finalize the integration plan and begin UAT preparation
  tone: formal
  audience: senior client stakeholders
  output_format: structured paragraph with clear sections

----------------------------------------------------------------------
Generated Prompt:
----------------------------------------------------------------------
Write a concise weekly status summary for the DeltaFin project.

Summarize progress made during Week 4, note testing delays caused by the external vendor's infrastructure issues, and outline finalize the integration plan and begin UAT preparation
in a formal tone suitable for senior client stakeholders.

Additional context:
- Progress: Completed API integration testing and resolved 1

### Test Case 2: MediTrack Project (Healthcare Technology)

In [9]:
# Define variables for Project 2: MediTrack
project2_vars = {
    "client_name": "MediTrack",
    "time_period": "Sprint 2",
    "progress": "Successfully deployed the patient data dashboard and completed security audit",
    "blockers": "pending approvals from the compliance team for HIPAA certification",
    "next_steps": "complete the data-mapping activity and prepare for pilot testing",
    "tone": "neutral, business-professional",
    "audience": "internal project sponsors",
    "output_format": "bullet-point list with clear categories"
}

# Substitute variables into the template
project2_prompt = prompt_template.format(**project2_vars)

print("🟣 TASK 5: Testing Template - Project 2 (MediTrack)")
print("="*70)
print("\nVariable Values:")
for key, value in project2_vars.items():
    print(f"  {key}: {value}")

print("\n" + "-"*70)
print("Generated Prompt:")
print("-"*70)
print(project2_prompt)

print("\n" + "-"*70)
print("Generating output...\n")

# Generate output
project2_output = llm.invoke(project2_prompt)

print("Generated Status Summary:")
print("-"*70)
print(project2_output)
print("-"*70)

🟣 TASK 5: Testing Template - Project 2 (MediTrack)

Variable Values:
  client_name: MediTrack
  time_period: Sprint 2
  progress: Successfully deployed the patient data dashboard and completed security audit
  blockers: pending approvals from the compliance team for HIPAA certification
  next_steps: complete the data-mapping activity and prepare for pilot testing
  tone: neutral, business-professional
  audience: internal project sponsors
  output_format: bullet-point list with clear categories

----------------------------------------------------------------------
Generated Prompt:
----------------------------------------------------------------------
Write a concise weekly status summary for the MediTrack project.

Summarize progress made during Sprint 2, note pending approvals from the compliance team for HIPAA certification, and outline complete the data-mapping activity and prepare for pilot testing
in a neutral, business-professional tone suitable for internal project sponsors.



### Task 5: Conclusion

**Testing Results:**
- ✅ Template produces consistent, high-quality outputs
- ✅ Successfully adapts to different contexts
- ✅ Maintains appropriate tone for each audience
- ✅ Generates properly formatted summaries

**Template Benefits:**
1. **Time Savings**: Analysts don't rewrite prompts from scratch
2. **Consistency**: All outputs follow the same structure
3. **Quality**: Professional standards maintained across team
4. **Scalability**: Works for unlimited number of projects
5. **Flexibility**: Adapts to different tones and audiences

**Conclusion:** Your template now produces clear, consistent outputs for any project in your team. You've successfully created a scalable, reusable prompt template that analysts across the organization can use.